# Human-in-the-Loop — Step-by-Step Tests

Tests each step of the interrupt/resume implementation as it is built.

- **Step 1** — Form schema validation (`forms.py`)
- **Step 2** — `HumanInputNode` interrupt/resume cycle (minimal graph)
- **Step 3** — Orchestrator routing to `human_input` node
- **Step 4** — SSE `interrupt` event detection via API
- **Step 5** — `/api/request/resume` endpoint round-trip

In [3]:
import os
import sys

sys.path.insert(0, os.path.abspath('../..'))

from apps.agentic.core.utils import set_anthropic_env

set_anthropic_env(filedir='../../.keys')

## Step 1 — Form Schema Validation

Verify that `LoadResearchDocumentForm` and `LoadGitHubRepoForm` accept valid data,
reject missing required fields, and that `validate_form()` routes by `type` correctly.

In [4]:
from apps.agentic.agents.forms import (
    LoadResearchDocumentForm,
    LoadGitHubRepoForm,
    LoadPDFDocumentForm,
    validate_form,
    FORM_REGISTRY,
)

print('Registered form types:', list(FORM_REGISTRY.keys()))

Registered form types: ['load_research_document', 'load_github_repo', 'load_pdf_document']


In [5]:
# Valid — all fields provided
form = LoadResearchDocumentForm(
    filename='jaynes_prob_theory.pdf',
    title='Probability Theory: The Logic of Science',
    authors='E.T. Jaynes',
    document_date='2003-06-09',
    topic='probability theory',
    shelf='publications',
)
print('Valid form:', form)
print()

Valid form: type='load_research_document' filename='jaynes_prob_theory.pdf' title='Probability Theory: The Logic of Science' authors='E.T. Jaynes' document_date='2003-06-09' topic='probability theory' shelf='publications'



In [7]:
# Invalid — missing required fields
from pydantic import ValidationError

try:
    LoadResearchDocumentForm(filename='jaynes_prob_theory.pdf')
except ValidationError as e:
    print('Expected ValidationError — missing fields:')
    for err in e.errors():
        print(f"  {err['loc'][0]}: {err['msg']}")

Expected ValidationError — missing fields:
  title: Field required
  authors: Field required
  document_date: Field required
  topic: Field required
  shelf: Field required


In [8]:
# Invalid — all fields missing
try:
    LoadResearchDocumentForm()
except ValidationError as e:
    print('Expected ValidationError — all fields missing:')
    for err in e.errors():
        print(f"  {err['loc'][0]}: {err['msg']}")

Expected ValidationError — all fields missing:
  filename: Field required
  title: Field required
  authors: Field required
  document_date: Field required
  topic: Field required
  shelf: Field required


In [9]:
# Valid GitHub repo form
form = LoadGitHubRepoForm(account='troystribling', repo='yada')
print('GitHub form:', form)
print()

GitHub form: type='load_github_repo' account='troystribling' repo='yada'



In [10]:
# Invalid — missing both required fields
try:
    LoadGitHubRepoForm()
except ValidationError as e:
    print('Expected ValidationError:')
    for err in e.errors():
        print(f"  {err['loc'][0]}: {err['msg']}")

Expected ValidationError:
  account: Field required
  repo: Field required


In [11]:
# validate_form — research document via dict with all fields
form = validate_form({
    'type': 'load_research_document',
    'filename': 'jaynes_prob_theory.pdf',
    'title': 'Probability Theory: The Logic of Science',
    'authors': 'E.T. Jaynes',
    'document_date': '2003-06-09',
    'topic': 'probability theory',
    'shelf': 'publications',
})
print('validate_form result:', form)
print()

validate_form result: type='load_research_document' filename='jaynes_prob_theory.pdf' title='Probability Theory: The Logic of Science' authors='E.T. Jaynes' document_date='2003-06-09' topic='probability theory' shelf='publications'



In [12]:
# validate_form — GitHub repo via dict
form = validate_form({
    'type': 'load_github_repo',
    'account': 'troystribling',
    'repo': 'yada',
})
print('validate_form result:', form)
print()

validate_form result: type='load_github_repo' account='troystribling' repo='yada'



In [13]:
# validate_form — unknown type
try:
    validate_form({'type': 'unknown_form'})
except ValueError as e:
    print('Expected ValueError:', e)

Expected ValueError: Unknown form type 'unknown_form'. Available: ['load_research_document', 'load_github_repo', 'load_pdf_document']


In [14]:
# validate_form — missing type key
try:
    validate_form({'filename': 'no_type.pdf'})
except ValueError as e:
    print('Expected ValueError:', e)

Expected ValueError: form_data must include a 'type' field


In [15]:
# Valid PDF document form
form = LoadPDFDocumentForm(
    filename='jaynes_prob_theory.pdf',
    title='Probability Theory: The Logic of Science',
    authors='E.T. Jaynes',
    published_date='2003-06-09',
    topic='probability theory',
    shelf='publications',
)
print('Valid PDF form:', form)
print()

Valid PDF form: type='load_pdf_document' filename='jaynes_prob_theory.pdf' title='Probability Theory: The Logic of Science' authors='E.T. Jaynes' published_date='2003-06-09' topic='probability theory' shelf='publications'



In [16]:
# Invalid PDF form — missing required fields
try:
    LoadPDFDocumentForm(filename='jaynes_prob_theory.pdf')
except ValidationError as e:
    print('Expected ValidationError — missing fields:')
    for err in e.errors():
        print(f"  {err['loc'][0]}: {err['msg']}")

Expected ValidationError — missing fields:
  title: Field required
  authors: Field required
  published_date: Field required
  topic: Field required
  shelf: Field required


In [17]:
# validate_form — PDF document via dict
form = validate_form({
    'type': 'load_pdf_document',
    'filename': 'jaynes_prob_theory.pdf',
    'title': 'Probability Theory: The Logic of Science',
    'authors': 'E.T. Jaynes',
    'published_date': '2003-06-09',
    'topic': 'probability theory',
    'shelf': 'publications',
})
print('validate_form result:', form)
print()

validate_form result: type='load_pdf_document' filename='jaynes_prob_theory.pdf' title='Probability Theory: The Logic of Science' authors='E.T. Jaynes' published_date='2003-06-09' topic='probability theory' shelf='publications'



## Step 2 — HumanInputNode Interrupt/Resume

Build a minimal 2-node graph (`trigger → human_input`) to test the interrupt/resume
cycle in isolation, before wiring into the orchestrator.

The `trigger` node injects a `form_type` into the last message's `additional_kwargs`
to signal which form to request. `HumanInputNode` then interrupts, and we resume
with dummy form data to confirm the cycle completes and validates correctly.

In [18]:
import shortuuid
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

from apps.agentic.core.agents.messages import WorkerState
from apps.agentic.core.agents.human_input_node import HumanInputNode
from apps.agentic.core.checkpointer import checkpointer


def trigger_node(state: WorkerState) -> WorkerState:
    """Simulates the orchestrator deciding a form is needed."""
    return {
        "messages": [
            AIMessage(
                content="I need some information to load the document.",
                additional_kwargs={"form_type": "load_research_document"},
            )
        ]
    }


graph = (
    StateGraph(WorkerState)
    .add_node("trigger", trigger_node)
    .add_node("human_input", HumanInputNode())
    .add_edge(START, "trigger")
    .add_edge("trigger", "human_input")
    .add_edge("human_input", END)
).compile(checkpointer=checkpointer)

print('Graph compiled.')

Graph compiled.


In [19]:
# First invocation — ainvoke returns early when interrupted (no exception raised)
import json

thread_id = shortuuid.uuid()
config = RunnableConfig(configurable={"thread_id": thread_id})

await graph.ainvoke(
    {"messages": [HumanMessage(content="Load a research document.")]},
    config,
)

# Inspect suspended state
state = await graph.aget_state(config)

interrupted = bool(state.tasks and state.tasks[0].interrupts)
print('Interrupted (expected True):', interrupted)
print()

# Extract the interrupt value (form schema) from the pending task
interrupt_value = state.tasks[0].interrupts[0].value
print('Form schema from interrupt:')
print(json.dumps(interrupt_value, indent=2))

DEBUG:    HumanInputNode: interrupting for form 'load_research_document'
Interrupted (expected True): True

Form schema from interrupt:
{
  "type": "load_research_document",
  "fields": {
    "filename": {
      "description": "Filename of the document to load.",
      "required": true
    },
    "title": {
      "description": "Document title.",
      "required": true
    },
    "authors": {
      "description": "Document authors.",
      "required": true
    },
    "document_date": {
      "description": "Publication or document date.",
      "required": true
    },
    "topic": {
      "description": "Topic or subject area.",
      "required": true
    },
    "shelf": {
      "description": "Library shelf to assign the document to.",
      "required": true
    }
  }
}


In [20]:
# Resume with valid form data — graph should complete and validate the form
resume_data = {
    'filename': 'jaynes_prob_theory.pdf',
    'title': 'Probability Theory: The Logic of Science',
    'authors': 'E.T. Jaynes',
    'document_date': '2003-06-09',
    'topic': 'probability theory',
    'shelf': 'publications',
}

result = await graph.ainvoke(Command(resume=resume_data), config)

last_msg = result['messages'][-1]
print('Last message type:', type(last_msg).__name__)
print('form_type:', last_msg.additional_kwargs.get('form_type'))
print('form_data:', last_msg.additional_kwargs.get('form_data'))

DEBUG:    HumanInputNode: interrupting for form 'load_research_document'
DEBUG:    HumanInputNode: resumed with data {'filename': 'jaynes_prob_theory.pdf', 'title': 'Probability Theory: The Logic of Science', 'authors': 'E.T. Jaynes', 'document_date': '2003-06-09', 'topic': 'probability theory', 'shelf': 'publications'}
Last message type: HumanMessage
form_type: load_research_document
form_data: {'type': 'load_research_document', 'filename': 'jaynes_prob_theory.pdf', 'title': 'Probability Theory: The Logic of Science', 'authors': 'E.T. Jaynes', 'document_date': '2003-06-09', 'topic': 'probability theory', 'shelf': 'publications'}


In [21]:
# Resume with invalid form data — should raise ValidationError
thread_id_2 = shortuuid.uuid()
config_2 = RunnableConfig(configurable={"thread_id": thread_id_2})

await graph.ainvoke(
    {"messages": [HumanMessage(content="Load a research document.")]},
    config_2,
)

try:
    result = await graph.ainvoke(Command(resume={'filename': 'missing_other_fields.pdf'}), config_2)
    print('Completed without error (unexpected):', result)
except ValidationError as e:
    print('Expected ValidationError on resume:')
    for err in e.errors():
        print(f"  {err['loc'][0]}: {err['msg']}")

DEBUG:    HumanInputNode: interrupting for form 'load_research_document'
DEBUG:    HumanInputNode: interrupting for form 'load_research_document'
DEBUG:    HumanInputNode: resumed with data {'filename': 'missing_other_fields.pdf'}
Expected ValidationError on resume:
  title: Field required
  authors: Field required
  document_date: Field required
  topic: Field required
  shelf: Field required


## Step 3 — Orchestrator Routing

*Cells will be added here after Step 3 is implemented.*

## Step 4 — SSE Interrupt Event

*Cells will be added here after Step 4 is implemented.*

## Step 5 — Resume Endpoint Round-Trip

*Cells will be added here after Step 5 is implemented.*